In [0]:
pip install xgboost

In [0]:
# ===============================
# XGBoost (time-aware, TARGET ENC)
# ===============================
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.functions import vector_to_array
from pyspark.mllib.evaluation import BinaryClassificationMetrics
import datetime as dt, builtins, math

# -----------------
# Config (edit here)
# -----------------
TABLE_NAME   = "hive_metastore.default.lreg_df"
DATE_COL     = "DATE"
LABEL_COL    = "has_falha"
ID_COL       = "ID"
LOC_COL      = "CONCELHO"
USE_WEATHER  = True   # flip to False for "no weather"

WEATHER_COLS = ["temperature", "precipitation", "wind_speed", "humidity"]
BASE_NUMERIC = [
    "INTENSITY","TENSION","H_LIM_I","H_LIM_T",
    "MAVERAGE_2H_I","MAVERAGE_2H_T","MAVERAGE_1D_I","MAVERAGE_1D_T",
    "EVENT_COUNT_I","EVENT_COUNT_T",
    "TIME_OVER_LIMIT_I","TIME_OVER_LIMIT_T",
    "DAY_OF_WEEK","DAY_OF_MONTH","DAY_OF_YEAR","HOUR_OF_DAY",
    "AMPM_flag"  # keep for stable layout
]

# Optional: very small feature lift (short lags + cyclical calendar)
from pyspark.sql import Window
w = Window.partitionBy(ID_COL).orderBy(F.col(DATE_COL))
for c in ["INTENSITY","TENSION"]:
    for k in [2,3,4,8,12]:  # 30m..3h for 15-min cadence; adjust if needed
        BASE_NUMERIC.append(f"{c}_lag{k}")
for c in ["INTENSITY","TENSION"]:
    for k in [4,8,24]:      # 1h,2h,6h rolling mean
        BASE_NUMERIC.append(f"{c}_rmean{k}")

final_df = spark.read.table(TABLE_NAME)




# Ensure AMPM_flag exists (idempotent)
if "AM_PM" in final_df.columns and "AMPM_flag" not in final_df.columns:
    final_df = final_df.withColumn("AMPM_flag", F.when(F.col("AM_PM")=="PM", 1.0).otherwise(0.0))

# Build the extra features if missing
for c in ["INTENSITY","TENSION"]:
    for k in [2,3,4,8,12]:
        coln = f"{c}_lag{k}"
        if coln not in final_df.columns:
            final_df = final_df.withColumn(coln, F.lag(F.col(c), k).over(w))
for c in ["INTENSITY","TENSION"]:
    for k in [4,8,24]:
        coln = f"{c}_rmean{k}"
        if coln not in final_df.columns:
            final_df = final_df.withColumn(coln, F.avg(F.col(c)).over(w.rowsBetween(-k+1, 0)))

# Cyclical calendar features
final_df = (final_df
    .withColumn("HOUR_SIN", F.sin(2*math.pi*F.col("HOUR_OF_DAY")/24.0))
    .withColumn("HOUR_COS", F.cos(2*math.pi*F.col("HOUR_OF_DAY")/24.0))
    .withColumn("DOW_SIN",  F.sin(2*math.pi*F.col("DAY_OF_WEEK")/7.0))
    .withColumn("DOW_COS",  F.cos(2*math.pi*F.col("DAY_OF_WEEK")/7.0))
)
for c in ["HOUR_SIN","HOUR_COS","DOW_SIN","DOW_COS"]:
    if c not in BASE_NUMERIC: BASE_NUMERIC.append(c)

# ---- EXTRAS: margins/ratios, spikes vs rolling means, simple weather deltas ----

# 1) margins / ratios vs high limits
final_df = (final_df
  .withColumn("I_margin",  F.col("H_LIM_I") - F.col("INTENSITY"))
  .withColumn("T_margin",  F.col("H_LIM_T") - F.col("TENSION"))
  .withColumn("I_ratio",   F.when(F.col("H_LIM_I") != 0, F.col("INTENSITY")/F.col("H_LIM_I")).otherwise(F.lit(0.0)))
  .withColumn("T_ratio",   F.when(F.col("H_LIM_T") != 0, F.col("TENSION")  /F.col("H_LIM_T")).otherwise(F.lit(0.0)))
  .withColumn("any_over",  ((F.col("INTENSITY")>F.col("H_LIM_I")) | (F.col("TENSION")>F.col("H_LIM_T"))).cast("int"))
)

# 2) spikes vs rolling means (if rmean exists; else 0.0 to keep shape)
for base, r4, r8, r24 in [
    ("INTENSITY","INTENSITY_rmean4","INTENSITY_rmean8","INTENSITY_rmean24"),
    ("TENSION","TENSION_rmean4","TENSION_rmean8","TENSION_rmean24")
]:
    final_df = (final_df
      .withColumn(f"{base}_spike4",  F.when(F.col(r4).isNotNull(),  F.col(base)-F.col(r4)).otherwise(F.lit(0.0)))
      .withColumn(f"{base}_spike8",  F.when(F.col(r8).isNotNull(),  F.col(base)-F.col(r8)).otherwise(F.lit(0.0)))
      .withColumn(f"{base}_spike24", F.when(F.col(r24).isNotNull(), F.col(base)-F.col(r24)).otherwise(F.lit(0.0)))
    )

# 3) simple weather deltas (optional)
if all(c in final_df.columns for c in ["temperature","wind_speed"]):
    w_id = Window.partitionBy(ID_COL).orderBy(F.col(DATE_COL))
    final_df = (final_df
      .withColumn("dtemp1", F.coalesce(F.col("temperature") - F.lag("temperature").over(w_id), F.lit(0.0)))
      .withColumn("dwind1", F.coalesce(F.col("wind_speed") - F.lag("wind_speed").over(w_id), F.lit(0.0)))
    )
else:
    final_df = final_df.withColumn("dtemp1", F.lit(0.0)).withColumn("dwind1", F.lit(0.0))

# extend BASE_NUMERIC with the new features (avoid duplicates)
for c in [
    "I_margin","T_margin","I_ratio","T_ratio","any_over",
    "INTENSITY_spike4","INTENSITY_spike8","INTENSITY_spike24",
    "TENSION_spike4","TENSION_spike8","TENSION_spike24",
    "dtemp1","dwind1"
]:
    if c not in BASE_NUMERIC:
        BASE_NUMERIC.append(c)

display(final_df.limit(5))


display(final_df.limit(5))

# -----------------------------------------
# Time split: 80/20 by whole days (no leak)
# -----------------------------------------
mm = (final_df
      .agg(F.to_date(F.min(DATE_COL)).alias("min_day"),
           F.to_date(F.max(DATE_COL)).alias("max_day"))
      .first())
min_day, max_day = mm["min_day"], mm["max_day"]
total_days = (max_day - min_day).days + 1
cutoff_day = min_day + dt.timedelta(days=int(total_days*0.8) - 1)

train_df = final_df.filter(F.to_date(F.col(DATE_COL)) <= F.lit(cutoff_day.isoformat())).cache()
test_df  = final_df.filter(F.to_date(F.col(DATE_COL))  > F.lit(cutoff_day.isoformat())).cache()
print(f"Range: {min_day} → {max_day} | cutoff (train≤) {cutoff_day}")
print("Rows — train:", train_df.count(), " | test:", test_df.count())
display(train_df.limit(5))

# --------------------------------------------------------
# Validation slice inside TRAIN: last 10% days as is_val
# --------------------------------------------------------
trmm = (train_df
        .agg(F.to_date(F.min(DATE_COL)).alias("tr_min"),
             F.to_date(F.max(DATE_COL)).alias("tr_max"))
        .first())
tr_min, tr_max = trmm["tr_min"], trmm["tr_max"]
tr_days = (tr_max - tr_min).days + 1
val_days = builtins.max(1, int(tr_days * 0.1))
val_start = tr_max - dt.timedelta(days=val_days - 1)

train_tr = train_df.filter(F.to_date(F.col(DATE_COL)) <  F.lit(val_start.isoformat())).cache()
train_va = train_df.filter(F.to_date(F.col(DATE_COL)) >= F.lit(val_start.isoformat())).cache()

train_flag = (train_df
    .withColumn("is_val", (F.to_date(F.col(DATE_COL)) >= F.lit(val_start.isoformat())))
    .withColumn("is_val", F.when(F.col("is_val").isNull(), F.lit(False)).otherwise(F.col("is_val")).cast(BooleanType()))
    .cache()
)

print(f"Train_tr: {tr_min} → {val_start - dt.timedelta(days=1)} | Train_va: {val_start} → {tr_max}")
print("Rows — train_tr:", train_tr.count(), " | train_va:", train_va.count())

# ---------------------------------------------
# Class imbalance weight from TRAIN_TR (only)
# ---------------------------------------------
pos_neg = train_tr.agg(
    F.sum(F.col(LABEL_COL).cast("int")).alias("pos"),
    (F.count("*") - F.sum(F.col(LABEL_COL).cast("int"))).alias("neg")
).first()
scale_pos_weight = (pos_neg["neg"] / builtins.max(1, pos_neg["pos"])) if pos_neg["pos"] else 1.0
print("scale_pos_weight =", scale_pos_weight)

# ---------------------------------------------
# Target Encoding (smoothed, from TRAIN_TR only)
# ---------------------------------------------
# m-estimate smoothing: te = (sum_y + m*global_mean) / (count + m)
GLOBAL_MEAN = train_tr.agg(F.avg(F.col(LABEL_COL).cast("double"))).first()[0]
m = 50.0  # smoothing strength; tune 20..200

id_stats = (train_tr.groupBy(ID_COL)
            .agg(F.sum(F.col(LABEL_COL).cast("double")).alias("sum_y"),
                 F.count("*").alias("cnt")))
id_te = id_stats.withColumn("ID_te", (F.col("sum_y") + F.lit(m*GLOBAL_MEAN)) / (F.col("cnt") + F.lit(m))) \
                .select(ID_COL, "ID_te")

loc_stats = (train_tr.groupBy(LOC_COL)
             .agg(F.sum(F.col(LABEL_COL).cast("double")).alias("sum_y"),
                  F.count("*").alias("cnt")))
loc_te = loc_stats.withColumn("CONCELHO_te", (F.col("sum_y") + F.lit(m*GLOBAL_MEAN)) / (F.col("cnt") + F.lit(m))) \
                  .select(LOC_COL, "CONCELHO_te")

def apply_te(df):
    df = (df.join(id_te, on=ID_COL, how="left")
            .join(loc_te, on=LOC_COL, how="left")
            .fillna({"ID_te": GLOBAL_MEAN, "CONCELHO_te": GLOBAL_MEAN}))
    return df

train_flag_te = apply_te(train_flag).cache()
train_va_te   = apply_te(train_va).cache()
test_df_te    = apply_te(test_df).cache()

# ----------------------------
# Build XGB pipeline (NO OHE/SCALER)
# ----------------------------
NUMERIC_COLS = [c for c in BASE_NUMERIC if c in final_df.columns] + \
               ([c for c in WEATHER_COLS if c in final_df.columns] if USE_WEATHER else [])
# add target encodings
NUMERIC_COLS = list(dict.fromkeys(NUMERIC_COLS + ["ID_te","CONCELHO_te"]))  # unique, keep order

def make_pipeline_xgb(
    learning_rate=0.06, max_depth=6, subsample=0.8, colsample_bytree=0.6,
    num_round=1200, reg_alpha=0.0, reg_lambda=2.0, min_child_weight=5, gamma=1.0,
    scale_pos_weight=None    # <-- add this
):
    feats = VectorAssembler(inputCols=NUMERIC_COLS, outputCol="features",
                            handleInvalid="keep")  # keep this
    xgb = SparkXGBClassifier(
        features_col="features",
        label_col=LABEL_COL,
        prediction_col="prediction",
        probability_col="probability",
        raw_prediction_col="rawPrediction",
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        num_round=num_round,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        min_child_weight=min_child_weight,
        gamma=gamma,
        eval_metric="aucpr",
        tree_method="hist",
        num_workers=builtins.max(1, spark.sparkContext.defaultParallelism // 2),
        validation_indicator_col="is_val",
        early_stopping_rounds=100
    )
    if scale_pos_weight is not None:
        xgb.setParams(scale_pos_weight=float(scale_pos_weight))  # mutate in-place; DO NOT reassign


    return Pipeline(stages=[feats, xgb])

# ---------------------------
# Eval helpers (AUPRC/ROC)
# ---------------------------
def eval_binary(pred_df, label_col=LABEL_COL, prob_col="probability", thr=0.5, title=None):
    scored = (pred_df
              .withColumn("p_arr", vector_to_array(F.col(prob_col)))
              .withColumn("p1", F.col("p_arr")[1])
              .withColumn("pred_thr", (F.col("p1") >= F.lit(thr)).cast("int")))
    rdd = scored.select(F.col("p1").cast("double"), F.col(label_col).cast("double")).rdd.map(tuple)
    m = BinaryClassificationMetrics(rdd)
    auprc, auroc = m.areaUnderPR, m.areaUnderROC

    tp = scored.filter((F.col(label_col)==1) & (F.col("pred_thr")==1)).count()
    fp = scored.filter((F.col(label_col)==0) & (F.col("pred_thr")==1)).count()
    fn = scored.filter((F.col(label_col)==1) & (F.col("pred_thr")==0)).count()
    tn = scored.filter((F.col(label_col)==0) & (F.col("pred_thr")==0)).count()
    precision = tp/(tp+fp) if (tp+fp) else 0.0
    recall    = tp/(tp+fn) if (tp+fn) else 0.0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0

    out = spark.createDataFrame([(thr, precision, recall, f1, tp, fp, fn, tn, auprc, auroc)],
                                ["threshold","precision","recall","f1","TP","FP","FN","TN","AUPRC","AUROC"])
    if title: print(title)
    display(out)
    return {"AUPRC": auprc, "AUROC": auroc, "precision": precision, "recall": recall, "f1": f1}

def sweep_thresholds(pred_df, thresholds=(0.9,0.85,0.8,0.75,0.7,0.65,0.6)):
    rows = []
    for t in thresholds:
        s = eval_binary(pred_df, thr=t)
        rows.append((t, s["precision"], s["recall"], s["f1"], s["AUPRC"], s["AUROC"]))
    out = spark.createDataFrame(rows, ["threshold","precision","recall","f1","AUPRC","AUROC"])
    display(out.orderBy(F.desc("threshold")))
    return out

# ---------------------------------------
# Manual, time-aware param sweep (AUPRC)
# ---------------------------------------
search_space = [
    # (learning_rate, max_depth, colsample_bytree, min_child_weight, gamma, reg_lambda, num_round)
    (0.10, 6, 0.6, 5, 1.0, 2.0, 800),
    (0.06, 6, 0.6, 10, 1.0, 2.0, 1200),
    (0.05, 6, 0.8, 8,  0.0, 3.0, 1500),
    (0.04, 8, 0.6, 12, 1.0, 3.0, 2000),
]

best = {"auprc": -1, "cfg": None, "model": None}
for cfg in search_space:
    lr, md, csbt, mcw, gma, rl2, nr = cfg
    print(f"Training cfg: lr={lr} md={md} csbt={csbt} mcw={mcw} gamma={gma} regL2={rl2} rounds={nr}")
    pipe = make_pipeline_xgb(
        learning_rate=lr, max_depth=md, colsample_bytree=csbt,
        min_child_weight=mcw, gamma=gma, reg_lambda=rl2, num_round=nr
    )
    model = pipe.fit(train_flag_te)  # uses boolean is_val for early stopping
    pred_va = model.transform(train_va_te).cache()
    m = eval_binary(pred_va, thr=0.5, title="Validation @0.5")
    if m["AUPRC"] > best["auprc"]:
        best = {"auprc": m["AUPRC"], "cfg": cfg, "model": model}

print("Best by AUPRC on validation:", best["cfg"], "AUPRC:", best["auprc"])

# ---------------------------
# Final test evaluation
# ---------------------------
best_model = best["model"]
pred_test = best_model.transform(test_df_te).cache()
display(pred_test.select("probability","prediction", LABEL_COL).limit(10))

print("=== TEST @0.5 ===")
test_m = eval_binary(pred_test, thr=0.5)
print("=== TEST threshold sweep ===")
sweep_thresholds(pred_test)

# ---------------------------
# Feature importance (gain)
# ---------------------------
def vector_attr_names(df_with_vec, vec_col="features"):
    meta = df_with_vec.schema[vec_col].metadata
    names = []
    if "ml_attr" in meta and "attrs" in meta["ml_attr"]:
        for k in ["binary","nominal","numeric"]:
            for a in meta["ml_attr"]["attrs"].get(k, []):
                names.append((a["idx"], a["name"]))
    return [n for _, n in sorted(names, key=lambda x: x[0])]

booster = best_model.stages[-1].get_booster()
score = booster.get_score(importance_type="gain")  # dict like {"f0": gain, ...}

# Use whatever training DF you actually have in this run
sample_df = train_flag_te if 'train_flag_te' in locals() else train_df
tmp_feat = best_model.transform(sample_df.limit(1)).select("features")
feat_names = vector_attr_names(tmp_feat, "features")

rows = []
for i, name in enumerate(feat_names):
    rows.append((i, name, float(score.get(f"f{i}", 0.0))))
imp = spark.createDataFrame(rows, ["idx","feature","gain"]).orderBy(F.desc("gain"))
print("Top 25 features (gain)")
display(imp.limit(25))


In [0]:
# choose threshold by maximizing F2 (recall-weighted)
def sweep_thresholds_fbeta(pred_df, beta=2.0, steps=101):
    import numpy as np
    from pyspark.sql import functions as F
    from pyspark.ml.functions import vector_to_array
    pdf = (pred_df
           .withColumn("p", vector_to_array("probability")[1].cast("double"))
           .select("p", F.col(LABEL_COL).cast("double").alias("y"))
           .toPandas())
    ts = np.linspace(0.05, 0.95, steps)
    rows = []
    for t in ts:
        yhat = (pdf["p"] >= t).astype(int)
        TP = ((yhat==1)&(pdf["y"]==1)).sum()
        FP = ((yhat==1)&(pdf["y"]==0)).sum()
        FN = ((yhat==0)&(pdf["y"]==1)).sum()
        prec = TP/(TP+FP) if (TP+FP) else 0.0
        rec  = TP/(TP+FN) if (TP+FN) else 0.0
        fb   = (1+beta*beta)*prec*rec/((beta*beta)*prec+rec) if (prec+rec) else 0.0
        rows.append((float(t), float(prec), float(rec), float(fb)))
    import pandas as pd
    df = pd.DataFrame(rows, columns=["threshold","precision","recall",f"F{beta}"])
    t_star = df.sort_values(f"F{beta}", ascending=False).iloc[0]["threshold"]
    return t_star, df

# on validation predictions:
pred_va = best_model.transform(train_va_te).cache()
t_star, df_f2 = sweep_thresholds_fbeta(pred_va, beta=2.0)
print("Chosen threshold (max F2 on validation):", t_star)

# apply to TEST
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
pred_test = best_model.transform(test_df_te).cache()
scored = (pred_test
          .withColumn("p", vector_to_array("probability")[1].cast("double"))
          .withColumn("pred_thr", (F.col("p") >= F.lit(float(t_star))).cast("int")))
display(scored.select("p","pred_thr",LABEL_COL).limit(10))


In [0]:
# ==========================================================
# RECALL-OPTIMIZED XGBOOST SWEEP — SAFE v4 (setParams fix)
# ==========================================================

from pyspark.ml import Pipeline as _Pipeline
from pyspark.ml.feature import VectorAssembler as _VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from pyspark.mllib.evaluation import BinaryClassificationMetrics
import numpy as np, builtins, importlib, types

# Preconditions from your prep steps
assert 'NUMERIC_COLS' in locals(), "NUMERIC_COLS not found."
assert 'LABEL_COL' in locals(), "LABEL_COL not found."
assert 'train_flag_te' in locals(), "train_flag_te (with boolean is_val) not found."
assert 'train_va_te' in locals(), "train_va_te not found."
assert 'test_df_te' in locals(), "test_df_te not found."
if 'scale_pos_weight' not in locals():
    scale_pos_weight = 1.0

# ---------- Builder v4: importlib + setParams (no reassignment) ----------
def make_xgb_recall_pipeline_v4(
    train_like_df,
    numeric_cols,
    label_col,
    learning_rate=0.06, max_depth=6, subsample=0.8, colsample_bytree=0.6,
    num_round=1200, reg_alpha=0.0, reg_lambda=2.0,
    min_child_weight=5, gamma=1.0, scale_pos_weight=None
):
    # Validate features
    cols_in_df = set(train_like_df.columns)
    valid_cols = [c for c in numeric_cols if c in cols_in_df]
    missing = [c for c in numeric_cols if c not in cols_in_df]
    if missing:
        print(f"[builder] Dropping {len(missing)} missing cols (showing up to 10): {missing[:10]}")
    if not valid_cols:
        raise ValueError("[builder] No valid feature columns found in the dataframe.")

    feats = _VectorAssembler(inputCols=valid_cols, outputCol="features", handleInvalid="keep")

    xgb_mod = importlib.import_module('xgboost.spark')
    XGBClass = getattr(xgb_mod, 'SparkXGBClassifier', None)
    if XGBClass is None or not (isinstance(XGBClass, type) or isinstance(XGBClass, types.FunctionType)):
        raise RuntimeError("[builder] Could not obtain SparkXGBClassifier from xgboost.spark.")

    xgb = XGBClass(
        features_col="features",
        label_col=label_col,
        prediction_col="prediction",
        probability_col="probability",
        raw_prediction_col="rawPrediction",
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        num_round=num_round,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        min_child_weight=min_child_weight,
        gamma=gamma,
        eval_metric="aucpr",
        tree_method="hist",
        num_workers=builtins.max(1, spark.sparkContext.defaultParallelism // 2),
        validation_indicator_col="is_val",
        early_stopping_rounds=100
    )
    if scale_pos_weight is not None:
        # 👈 mutate in-place; DO NOT reassign
        xgb.setParams(scale_pos_weight=float(scale_pos_weight))

    pipe = _Pipeline(stages=[feats, xgb])

    st = pipe.getStages()
    print("[builder] Stage types:", [type(s).__name__ for s in st], "| XGB class:", XGBClass)
    if any(s is None for s in st):
        raise RuntimeError("[builder] Found None in pipeline stages (check setParams reassignment).")
    return pipe, valid_cols

# ---------- Helpers ----------
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
import numpy as np
from pyspark.mllib.evaluation import BinaryClassificationMetrics

def choose_threshold_fbeta(pred_df, label_col=LABEL_COL, beta=2.0, steps=101):
    pdf = (pred_df
           .withColumn("p", vector_to_array("probability")[1].cast("double"))
           .select("p", F.col(label_col).cast("double").alias("y"))
           .toPandas())
    ts = np.linspace(0.05, 0.95, steps)
    best = {"t": 0.5, "P": 0.0, "R": 0.0, "Fbeta": 0.0}
    for t in ts:
        yhat = (pdf["p"] >= t).astype(int)
        TP = int(((yhat==1)&(pdf["y"]==1)).sum())
        FP = int(((yhat==1)&(pdf["y"]==0)).sum())
        FN = int(((yhat==0)&(pdf["y"]==1)).sum())
        P  = TP/(TP+FP) if (TP+FP) else 0.0
        R  = TP/(TP+FN) if (TP+FN) else 0.0
        fbeta = (1+beta*beta)*P*R/((beta*beta)*P+R) if (P+R) else 0.0
        if fbeta > best["Fbeta"]:
            best = {"t": float(t), "P": float(P), "R": float(R), "Fbeta": float(fbeta)}
    return best

def pr_roc(pred_df, label_col=LABEL_COL):
    rdd = (pred_df
           .withColumn("p", vector_to_array("probability")[1].cast("double"))
           .select("p", F.col(label_col).cast("double"))
           .rdd.map(tuple))
    m = BinaryClassificationMetrics(rdd)
    return float(m.areaUnderPR), float(m.areaUnderROC)

# ---------- Recall-biased grids ----------
spw_base = scale_pos_weight
spw_candidates = sorted({float(spw_base),
                         float(spw_base*1.5),
                         float(spw_base*2.0),
                         float(spw_base*3.0)})

param_configs = [
    (0.06,  8, 0.70, 3, 0.0, 2.0, 1500),
    (0.05, 10, 0.70, 3, 0.0, 2.0, 2000),
    (0.08,  6, 0.60, 3, 0.0, 2.0, 1200),
    (0.05,  8, 0.80, 5, 0.0, 3.0, 1800),
]

# ---------- Sweep ----------
results = []
best = {"F2": -1.0, "model": None, "thr": 0.5, "cfg": None, "spw": None}

for spw in spw_candidates:
    for (lr, md, csbt, mcw, gma, rl2, nr) in param_configs:
        print(f"[FIT] spw={spw:.3f} | lr={lr} md={md} colsample={csbt} mcw={mcw} gamma={gma} regL2={rl2} rounds={nr}")

        pipe, used_cols = make_xgb_recall_pipeline_v4(
            train_like_df=train_flag_te,
            numeric_cols=NUMERIC_COLS,
            label_col=LABEL_COL,
            learning_rate=lr, max_depth=md, colsample_bytree=csbt,
            min_child_weight=mcw, gamma=gma, reg_lambda=rl2, num_round=nr,
            scale_pos_weight=spw
        )
        print(f"[FIT] Using {len(used_cols)} features.")

        model = pipe.fit(train_flag_te)                # early stopping via boolean is_val
        pred_va = model.transform(train_va_te).cache() # validation predictions

        best_val = choose_threshold_fbeta(pred_va, beta=2.0)
        thr = best_val["t"]; P = best_val["P"]; R = best_val["R"]; F2 = best_val["Fbeta"]
        auprc, auroc = pr_roc(pred_va)

        results.append((float(spw), lr, md, csbt, mcw, gma, rl2, nr, thr, P, R, F2, auprc, auroc))
        if F2 > best["F2"]:
            best = {"F2": F2, "model": model, "thr": thr,
                    "cfg": (lr, md, csbt, mcw, gma, rl2, nr), "spw": spw}

# ---------- Validation leaderboard ----------
schema = ["scale_pos_weight","lr","max_depth","colsample_bytree",
          "min_child_weight","gamma","reg_lambda","num_round",
          "thr_F2","val_precision","val_recall","val_F2","val_AUPRC","val_AUROC"]
val_board = spark.createDataFrame(results, schema)
display(val_board.orderBy(F.desc("val_F2")))

print(f"\nBest (by val F2): spw={best['spw']:.3f}, cfg={best['cfg']}, "
      f"thr={best['thr']:.3f}, F2={best['F2']:.4f}")

# ---------- TEST at chosen threshold ----------
best_model = best["model"]
pred_test = best_model.transform(test_df_te).cache()

t_star = float(best["thr"])
scored = (pred_test
          .withColumn("p", vector_to_array("probability")[1].cast("double"))
          .withColumn("pred_thr", (F.col("p") >= F.lit(t_star)).cast("int")))

TP = scored.filter((F.col(LABEL_COL)==1) & (F.col("pred_thr")==1)).count()
FP = scored.filter((F.col(LABEL_COL)==0) & (F.col("pred_thr")==1)).count()
FN = scored.filter((F.col(LABEL_COL)==1) & (F.col("pred_thr")==0)).count()
precision = TP/(TP+FP) if (TP+FP) else 0.0
recall    = TP/(TP+FN) if (TP+FN) else 0.0
f1        = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0
auprc_t, auroc_t = pr_roc(pred_test)

summary = spark.createDataFrame(
    [(t_star, float(precision), float(recall), float(f1), float(auprc_t), float(auroc_t),
      float(best["spw"]),) + tuple(best["cfg"])],
    ["threshold","precision","recall","f1","AUPRC","AUROC","scale_pos_weight",
     "lr","max_depth","colsample_bytree","min_child_weight","gamma","reg_lambda","num_round"]
)
print("=== TEST (at validation-chosen threshold) ===")
display(summary)


In [0]:
final_df = (final_df
    .withColumn("I_margin", F.col("INTENSITY") - F.col("H_LIM_I"))
    .withColumn("T_margin", F.col("TENSION")   - F.col("H_LIM_T"))
    .withColumn("I_ratio",  F.col("INTENSITY")/F.col("H_LIM_I"))
    .withColumn("T_ratio",  F.col("TENSION")  /F.col("H_LIM_T"))
)
BASE_NUMERIC += ["I_margin","T_margin","I_ratio","T_ratio"]


In [0]:
from pyspark.sql import Window
w = Window.partitionBy(ID_COL).orderBy(F.col(DATE_COL))
for c in ["INTENSITY","TENSION"]:
    if f"{c}_lag1" not in final_df.columns:
        final_df = final_df.withColumn(f"{c}_lag1", F.lag(F.col(c),1).over(w))
final_df = (final_df
    .withColumn("dI_1", F.col("INTENSITY")-F.col("INTENSITY_lag1"))
    .withColumn("dT_1", F.col("TENSION")  -F.col("TENSION_lag1"))
)
BASE_NUMERIC += ["dI_1","dT_1"]


In [0]:
over_i  = (F.col("INTENSITY") > F.col("H_LIM_I"))
over_t  = (F.col("TENSION")   > F.col("H_LIM_T"))
w_all   = Window.partitionBy(ID_COL).orderBy(F.col(DATE_COL)).rowsBetween(Window.unboundedPreceding, 0)

final_df = (final_df
    .withColumn("last_over_i_ts", F.max(F.when(over_i, F.col(DATE_COL))).over(w_all))
    .withColumn("last_over_t_ts", F.max(F.when(over_t, F.col(DATE_COL))).over(w_all))
    .withColumn("mins_since_over_i", (F.col("DATE").cast("long") - F.col("last_over_i_ts").cast("long"))/60.0)
    .withColumn("mins_since_over_t", (F.col("DATE").cast("long") - F.col("last_over_t_ts").cast("long"))/60.0)
    .drop("last_over_i_ts","last_over_t_ts")
)
BASE_NUMERIC += ["mins_since_over_i","mins_since_over_t"]


In [0]:
# After scoring, lift positives by a moving max over K steps (e.g., 4 = 1 hour if 15-min cadence)
from pyspark.sql import Window
K = 4
w_id = Window.partitionBy(ID_COL).orderBy(F.col(DATE_COL)).rowsBetween(-K+1, 0)

scored = (best_model.transform(test_df_te)
          .withColumn("p", vector_to_array("probability")[1].cast("double")))

scored = (scored
          .withColumn("p_smooth", F.max("p").over(w_id))
          .withColumn("pred_thr", (F.col("p_smooth") >= F.lit(float(t_star))).cast("int")))

# evaluate again with smoothed probs
display(scored.select("p","p_smooth","pred_thr",LABEL_COL).limit(10))
